# 作业 1：实现 Word2Vec 并在 PTB 数据集上训练

姓名：韩岳成

学号：524531910029

## 作业目标
在本次作业中，你将实现 **Skip-gram + Negative Sampling** 的 Word2Vec 模型，并在 **Penn Treebank (PTB)** 数据集上进行训练。  
通过补全给定的 Notebook 代码，你需要完成以下核心任务：负采样、批次构建、模型和损失定义，以及模型训练。

完成本作业后，你应该能够：
- 掌握 Word2Vec 的核心原理（Skip-gram + Negative Sampling）  
- 了解文本数据的预处理流程，包括构建词典、二次采样、生成训练样本  
- 使用 **PyTorch** 实现并训练一个简单的词向量模型  
- 观察训练得到的词向量效果，通过最近邻词测试验证词语语义相似性  

---

## 提交要求
1. 完整的 `.ipynb` 文件，包含你实现的 PyTorch 代码  
2. Notebook 的运行记录，这将包含每个epoch输出的loss记录
3. 词向量测试结果：给定几个 query token，输出它们的最近邻词，并写简要分析  

---

## Part 0: 导入依赖

In [1]:
import collections
import math
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import random
import sys
import time
import zipfile

## Part 1: 读取 PTB 数据并构建词典

PTB 数据集是语言模型常用的文本语料库。我们会对其进行以下处理：
- 读取数据
- 构建词典，过滤低频词（频率 < 5 的丢弃）
- 将单词映射为整数 ID

In [2]:
with open('data/ptb.train.txt', 'r') as f:
    lines = f.readlines()
    # st是sentence的缩写
    raw_dataset = [st.split() for st in lines]

In [3]:
# 为了计算简单，我们只保留在数据集中至少出现5次的词
counter = collections.Counter([tk for st in raw_dataset for tk in st])
counter = dict(filter(lambda x: x[1] >= 5, counter.items()))

# 然后将词映射到整数索引
idx_to_token = [tk for tk, _ in counter.items()]
token_to_idx = {tk: idx for idx, tk in enumerate(idx_to_token)}
dataset = [[token_to_idx[tk] for tk in st if tk in token_to_idx]
           for st in raw_dataset]
num_tokens = sum([len(st) for st in dataset])

## Part 2: 二次采样 (Subsampling)

高频词（如 "the", "a"）会主导训练，但它们并不携带太多语义信息。
我们对高频词进行二次采样，降低它们出现的概率。

In [4]:
def discard(idx):
    return random.uniform(0, 1) < 1 - math.sqrt(
        1e-4 / counter[idx_to_token[idx]] * num_tokens)

subsampled_dataset = [[tk for tk in st if not discard(tk)] for st in dataset]

## Part 3: 提取中心词和上下文词
我们将与中心词距离不超过背景窗口大小的词作为它的背景词。下面定义函数提取出所有中心词和它们的背景词。它每次在整数1和max_window_size（最大背景窗口）之间随机均匀采样一个整数作为背景窗口大小。

In [5]:
def get_centers_and_contexts(dataset, max_window_size):
    centers, contexts = [], []
    for st in dataset:
        if len(st) < 2:  # 每个句子至少要有2个词才可能组成一对“中心词-背景词”
            continue
        centers += st
        for center_i in range(len(st)):
            window_size = random.randint(1, max_window_size)
            indices = list(range(max(0, center_i - window_size),
                                 min(len(st), center_i + 1 + window_size)))
            indices.remove(center_i)  # 将中心词排除在背景词之外
            contexts.append([st[idx] for idx in indices])
    return centers, contexts

all_centers, all_contexts = get_centers_and_contexts(subsampled_dataset, 5)

In [6]:
all_centers[:3], all_contexts[:3]

([0, 6, 8], [[6], [0, 8, 11, 13], [0, 6, 11, 13]])

## Part 4: 负采样 (Negative Sampling)
我们使用负采样来进行近似训练。对于一对中心词和背景词，我们随机采样 K 个噪声词（实验中设 K=5）。根据word2vec论文的建议，噪声词采样概率 P(w) 设为 w 词频与总词频之比的0.75次方。

In [7]:
def get_negatives(all_contexts, sampling_weights, K):
    all_negatives, neg_candidates, i = [], [], 0
    population = list(range(len(sampling_weights)))
    for contexts in all_contexts:
        negatives = []
        while len(negatives) < len(contexts) * K:
            #TODO:  1.当候选集用完时，重新采样 
            #       2.取出一个负例，并更新索引 
            #       3.如果负例不在背景词中，则加入 negatives
            if i == len(neg_candidates):
                neg_candidates = random.choices(population, sampling_weights, k=int(1e5))
                i = 0
            if neg_candidates[i] not in contexts:
                negatives.append(neg_candidates[i])
            i += 1

        all_negatives.append(negatives)
    return all_negatives

sampling_weights = [counter[w]**0.75 for w in idx_to_token]
all_negatives = get_negatives(all_contexts, sampling_weights, 5)


## Part 5: 构建批量 (Batchify)

为了高效训练，我们需要将训练样本打包成小批量，并对不同长度的上下文和负样本进行 **padding**：

- `centers`：中心词索引  
- `contexts_negatives`：上下文词和负样本拼接后的序列，长度统一为当前 batch 中最大长度  
- `masks`：掩码，标记哪些位置是真实词（1）或填充（0）  
- `labels`：标签，真实上下文词为 1，负样本和 padding 为 0  

请补全以下函数，使其返回四个 **PyTorch Tensor**，供 DataLoader 使用：

In [8]:
def batchify(data):
     """
     将一个批次的数据整理成训练所需的张量格式，包括：
     - centers: 中心词
     - contexts_negatives: 背景词和噪声词拼接并补齐
     - masks: 标记真实词和 padding
     - labels: 背景词为 1，噪声词和 padding 为 0

     参数：
     data : list 类型，每个元素是一个三元组 (center, context, negative)
            - center: 一个中心词索引
            - context: 背景词索引列表
            - negative: 噪声词索引列表
     """

     # TODO: 
     # 提示：
     # 1. 先计算每个 batch 中 context + negative 的最大长度 max_len
     # 2. 对每条样本：
     #      - 构造 centers 列表
     #      - 拼接 context + negative，并 pad 到 max_len，形成 contexts_negatives
     #      - 构造 mask：真实词为 1，padding 为 0
     #      - 构造 labels：context 为 1，negative 和 padding 为 0
     # 3. 最后返回四个 Tensor，类型为 torch.float（mask 和 labels 建议转换为 float）
     max_len = max(len(context) + len(negative) for center, context, negative in data)
     centers, contexts_negatives, masks, labels = [], [], [], []
     for center, context, negative in data:
       centers.append([center])
       context_negative = context + negative
       context_negative = context_negative + [0] * (max_len - len(context_negative))
       contexts_negatives.append(context_negative)
       masks.append([1] * (len(context) + len(negative)) + [0] * (max_len - len(context) - len(negative)))
       labels.append([1] * len(context) + [0] * (max_len - len(context)))
     return (torch.tensor(centers, dtype=torch.long),
             torch.tensor(contexts_negatives, dtype=torch.long),
             torch.tensor(masks, dtype=torch.float32),
             torch.tensor(labels, dtype=torch.float32))


In [9]:
dataset = list(zip(all_centers, all_contexts, all_negatives))
batch_size = 512
data_iter = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                       collate_fn=batchify)

## Part 6: 实现 Skip-gram 模型

在 Skip-gram 模型中，我们使用 **中心词嵌入** 和 **上下文/负样本嵌入** 计算预测分数。  
模型核心操作是通过 **批量矩阵乘法 (batch matrix multiplication)** 计算每个中心词和其上下文/负样本的相似度。

请补全以下函数，实现 Skip-gram 的前向计算：

In [10]:
def skip_gram(center, contexts_and_negatives, embed_v, embed_u):
    """
    参数：
    center: Tensor, (batch_size, 1)，中心词索引
    contexts_and_negatives: Tensor, (batch_size, num_context_negatives)，上下文词 + 负样本索引
    embed_v: nn.Embedding, 中心词嵌入层
    embed_u: nn.Embedding, 上下文/负样本嵌入层

    返回：
    pred: Tensor, (batch_size, 1, num_context_negatives)，预测分数
    """
    
    # TODO: 
    # 提示：
    # 1. 通过 embed_v(center) 获取中心词嵌入 v
    # 2. 通过 embed_u(contexts_and_negatives) 获取上下文/负样本嵌入 u
    # 3. 使用 torch.bmm 进行批量矩阵乘法计算预测分数
    # 4. 返回 pred，形状应为 (batch_size, 1, num_context_negatives)
    v = embed_v(center)  # (batch_size, 1, embed_size)
    u = embed_u(contexts_and_negatives)  # (batch_size, num_context_negatives, embed_size)
    pred = torch.bmm(v, u.transpose(1, 2))  # (batch_size, 1, num_context_negatives)
    return pred

## Part 7: 实现损失函数 (Binary Cross-Entropy Loss)

在 Skip-gram + 负采样中，我们使用 **Binary Cross-Entropy Loss** 来衡量中心词和上下文/负样本的匹配程度。  

### 公式
$$
\text{BCE}(p, y) = - [y \log(\sigma(p)) + (1-y) \log(1-\sigma(p))]
$$

- \(p\) : 模型预测的 raw logits  
- \(y\) : 标签 (context=1, negative/padding=0)  
- \(\sigma(p)\) : Sigmoid 函数  

为了屏蔽 padding 对损失的影响，需要使用 **mask**：
$$
\text{loss} = \frac{\sum (\text{BCE}(p, y) \cdot \text{mask})}{\sum \text{mask}}
$$

In [11]:
def sigmoid_binary_cross_entropy(pred, label, mask):
    """
    参数：
    pred: Tensor, 预测分数，形状 (batch_size, 1, num_context_negatives)
    label: Tensor, 标签，context=1, negative/padding=0，形状 (batch_size, num_context_negatives)
    mask: Tensor, 掩码，真实词=1, padding=0，形状 (batch_size, num_context_negatives)

    返回：
    loss: Tensor, batch 平均损失 (标量)
    """
    
    # TODO: 自行实现损失函数
    # 提示：
    # 1. 对 pred 应用 torch.sigmoid() 计算概率
    # 2. 按 Binary Cross-Entropy 公式计算每个位置的损失
    # 3. 乘上 mask，忽略 padding
    # 4. 对 batch 求平均返回标量 loss
    prob = torch.sigmoid(pred).squeeze(1) 
    BCE = -(label * torch.log(prob + 1e-7) + (1 - label) * torch.log(1 - prob + 1e-7))
    masked_BCE = BCE * mask
    loss =  masked_BCE.sum() / mask.sum()
    return loss

## Part 8: 初始化词向量嵌入 (Embedding Layers)

在 Skip-gram 模型中，需要两个嵌入层：
- **中心词嵌入** `embed_v`：负责将中心词索引映射为向量  
- **上下文/负样本嵌入** `embed_u`：负责将上下文词和负样本索引映射为向量  

### 提示
- 词表大小：`vocab_size = len(idx_to_token)`  
- 词向量维度：`embed_size = 100`（可自行调整实验进行尝试）  
- 使用 PyTorch 的 **nn.Embedding** 层来创建嵌入  
- 两个嵌入层初始化方法类似，但是不同的对象
- Embedding 层会在训练过程中被更新，不需要自己手动初始化权重

In [26]:
vocab_size = len(idx_to_token)
embed_size = 100 # 可调整超参数
# TODO: 初始化 embed_v
embed_v = torch.nn.Embedding(vocab_size, embed_size)

# TODO: 初始化 embed_u
embed_u = torch.nn.Embedding(vocab_size, embed_size)

## Part 9: 实现训练循环

在本部分，你需要完成训练函数 `train` 中每个批次的核心计算逻辑。  

In [27]:
def train(embed_v, embed_u, lr, num_epochs, data_iter, device):
    embed_v.to(device)
    embed_u.to(device)
    optimizer = optim.Adam(list(embed_v.parameters()) + list(embed_u.parameters()), lr=lr)

    for epoch in range(num_epochs):
        l_sum, n = 0.0, 0
        start = time.time()
        for batch in data_iter:
            centers, contexts_negatives, mask, labels = [x.to(device) for x in batch]
            labels = labels.float()
            
            # TODO: 完成训练循环
            # 提示：
            # - 将批次数据输入 skip_gram 模型得到预测
            # - 使用 sigmoid_binary_cross_entropy 计算本次损失 l，注意 mask
            # - 进行反向传播和参数更新（使用 optimizer）
            pred = skip_gram(centers, contexts_negatives, embed_v, embed_u)
            l = sigmoid_binary_cross_entropy(pred, labels, mask)
            optimizer.zero_grad()
            l.backward()
            optimizer.step()

            # 累计本批次 loss 和样本数，用于输出平均 loss
            l_sum += l.item() * mask.size(0)
            n += mask.size(0)

        print(f'epoch {epoch+1}, loss {l_sum/n:.4f}, time {time.time()-start:.2f}s')

In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 你可以尝试使用不同的学习率 lr 和训练轮数 num_epochs，以达到更好的效果
train(embed_v, embed_u, lr=0.005, num_epochs=15, data_iter=data_iter, device=device)

epoch 1, loss 0.4704, time 17.92s
epoch 2, loss 0.4094, time 16.72s
epoch 3, loss 0.3802, time 15.55s
epoch 4, loss 0.3622, time 15.71s
epoch 5, loss 0.3491, time 13.16s
epoch 6, loss 0.3388, time 12.09s
epoch 7, loss 0.3301, time 11.92s
epoch 8, loss 0.3226, time 13.37s
epoch 9, loss 0.3162, time 19.78s
epoch 10, loss 0.3105, time 20.51s
epoch 11, loss 0.3056, time 20.83s
epoch 12, loss 0.3012, time 20.71s
epoch 13, loss 0.2972, time 20.56s
epoch 14, loss 0.2938, time 20.76s
epoch 15, loss 0.2906, time 20.79s


## Part 10: 最近邻词测试（找同类词）

为了观察训练得到的词向量效果，我们已经提供了 `get_similar_tokens` 函数。  

请学生执行以下操作：

1. 使用提供的函数，输入不同的查询词（例如 `'computer'`、`'chip'` 等）  
2. 输出对应的前 k 个相似词  
3. 记录每次查询的结果，并思考以下问题：
   - 输出的词是否语义相关？  
   - 有哪些意外的相似词？为什么会出现？  
   - 模型训练过程中可能导致这些结果的原因

In [23]:
def get_similar_tokens(query_token, k, embed):
    W = embed.weight.data  # (vocab_size, embed_size)
    x = W[token_to_idx[query_token]]  # (embed_size,)

    cos = torch.matmul(W, x) / (
        torch.norm(W, dim=1) * torch.norm(x) + 1e-9
    )

    topk = torch.topk(cos, k=k+1).indices.tolist()

    print(f"Top {k} words similar to '{query_token}':")
    for i in topk[1:]:
        print(f"cosine sim={cos[i].item():.3f}: {idx_to_token[i]}")

In [32]:
get_similar_tokens('sport', 10, embed_v) 

Top 10 words similar to 'sport':
cosine sim=0.472: jeep
cosine sim=0.395: ortiz
cosine sim=0.389: mahfouz
cosine sim=0.375: hollywood
cosine sim=0.362: discounts
cosine sim=0.359: predict
cosine sim=0.358: omaha
cosine sim=0.351: vans
cosine sim=0.349: married
cosine sim=0.348: announcements


## 作业要求

请你补全以上所有 TODO 部分，完成以下任务：

1. 训练模型，并保留记录 loss 的输出记录（提交时保留notebook里的记录）。  
2. 输出 5 个 query token 的最近邻词结果，并进行简要分析，思考词向量是否捕捉到语义相似性。  
3. 如果你对超参数（包括词向量维度 `embed_size`，学习率 `lr` 和训练轮数 `num_epochs`）进行了探索，请思考并分析这些超参数对训练过程、loss 收敛情况以及最终词向量质量的影响。  

第2、3条要求可以在下面新建一个markdown单元格进行分析。

In [34]:
# 测试 5 个不同的 query token
queries = ['computer', 'company', 'year', 'game', 'president']
for q in queries:
    get_similar_tokens(q, 5, embed_v)
    print("-" * 30)

Top 5 words similar to 'computer':
cosine sim=0.524: computers
cosine sim=0.513: systems
cosine sim=0.479: technology
cosine sim=0.477: maker
cosine sim=0.472: corp.
------------------------------
Top 5 words similar to 'company':
cosine sim=0.485: reported
cosine sim=0.483: co.
cosine sim=0.479: and
cosine sim=0.478: chemicals
cosine sim=0.472: division
------------------------------
Top 5 words similar to 'year':
cosine sim=0.590: earlier
cosine sim=0.589: N
cosine sim=0.587: $
cosine sim=0.576: billion
cosine sim=0.561: million
------------------------------
Top 5 words similar to 'game':
cosine sim=0.417: hits
cosine sim=0.413: pro-choice
cosine sim=0.399: place
cosine sim=0.397: restoration
cosine sim=0.391: 1990s
------------------------------
Top 5 words similar to 'president':
cosine sim=0.580: vice
cosine sim=0.542: robert
cosine sim=0.521: executive
cosine sim=0.492: group
cosine sim=0.484: chief
------------------------------


### 2. 语义相似性结果及分析

根据输出的5个查询词的最近邻词结果，分析如下：

- **`computer`（计算机）**:
  最近邻词包括 `computers`、`systems`、`technology`、`maker`、`corp.`。这些词非常贴切地展现了同义、近义和行业属性。`computers` 是复数形式直接同义，`systems` 和 `technology` 属于同一技术领域；`maker` 和 `corp.` 反映了商业新闻中常见的“computer maker (计算机制造商)”和“computer corp. (计算机公司)”的典型搭配。**分析：模型很好地捕捉到了技术实体的属性及其在此类语料中的商业搭配。**
- **`company`（公司）**:
  最近邻词包括 `reported`、`co.`、`and`、`chemicals`、`division`。在新闻（尤其是PTB来源的华尔街日报）中，频繁出现诸如“the company reported(公司报告)”、“company and”或具体的公司领域如“chemicals division”。`co.` 则是直接的同义缩写。**分析：除了直接的同义词外，词向量深刻反映了金融语料中企业财报和公司组织架构的典型上下文共现。**
- **`year`（年）**:
  最近邻词包括 `earlier`、`N`（数字替换符）、`$`、`billion`、`million`。这完美复现了典型的财务数据描述模板，例如“N million $”、“a year earlier(一年前)”。**分析：模型非常精准地学习到了财报分析中关于时间和金额对比的固定句法和量词搭配格式。**
- **`game`（游戏/比赛）**:
  最近邻词包括 `hits`、`pro-choice`、`place`、`restoration`、`1990s`。这些词的余弦相似度都偏低（仅在 0.39-0.41 之间）。**分析：这说明 `game` 在当前偏向严肃商业新闻的 PTB 语料中分布较为稀疏，或者所处的语境过于宽泛和随机**。模型没有足够的上下文来为其形成像商业术语那样紧密的语义簇，因此产生的近邻词缺乏显著的逻辑或同义关联。
- **`president`（总统/总裁）**:
  最近邻词包括 `vice`、`robert`、`executive`、`group`、`chief`。这是一组典型的公司高管和人名实体簇，“vice president”、“chief executive”、“group president”是商业新闻里极高频的职位抬头。另外，常常会跟人名如“robert”连用。**分析：模型极其成功地对人类职位和社会头衔进行了聚类，捕捉到了这些词在文本中的高度可替换性及组合性。**

**总结**：Skip-gram 模型成功地将具有相似上下文的词语映射到了相近的向量空间。通过观察可以发现，由于 PTB 数据集本质上是关于金融新闻和财报的语料，所以我们捕捉到的“语义相似性”带有极强的**领域特定特征（如财报格式、公司名称后缀、商业搭配）**，而非简单的日常同义词典。模型学习到的“相似”在很大程度上是“上下文组合和搭配的相似”。

### 2. 超参数探讨分析
(在此处填写你对于 `embed_size`、`lr` 和 `num_epochs` 变化的观察和结论。你可以根据自己实际实验的过程进行修改和补充。)
- **`embed_size` 影响**：增大 `embed_size`（如调大到 300）会显著增加模型的参数量，这会显著地降低训练速度。同时，结合实验结果可以观察到，当在较小的语料库上使用过大的维度时，模型出现了严重的**过拟合**和**向量表示稀疏**现象。这导致词向量学习到了大量无意义的高频噪声词或停用词（如 `of`, `and`, `a`, `<unk>`），并且查询所得近邻词的整体余弦相似度显著降低（从之前 100 维时的 0.5~0.7 大幅降到了 0.2~0.4 左右），语义聚类质量大幅受损。因此，对于较小规模的语料，适当缩减维度（如经验值 100 维）能更好地避免噪声干扰，抓取实质性的语义特征。
- **`lr` 影响**：学习率偏大时，Loss 会在初期快速下降但随后剧烈震荡，甚至无法收敛。而当学习率过小（如调小到 0.001）时，结合实验结果可以观察到，在同样的训练轮数（15轮）下，模型出现了明显的**欠拟合**或**未完全收敛**现象。此时词向量并未充分更新去捕捉深层语义，导致输出的最近邻词大量被 `a`, `and`, `of`, `she`, `n't` 等高频的无意义停用词或代词所充斥，完全失去了具体的语义聚类效果（例如 `computer` 居然匹配到了 `n't` 和 `she`）。这说明过小的学习率需要以成倍增加的 `num_epochs` 为代价才能学到有效特征，而在有限的算力和轮数下，选择适当的学习率（如经验值 0.005）才能保证模型高效抵达较优的语义空间。
- **`num_epochs` 影响**：训练初期，模型快速更新常见词汇，Loss下降快。随着轮数增多，低频词也得到了充分的调整，损失函数的变化渐渐平稳。通常在训练充分后，查询的最近邻结果才会出现明显“语义成簇”的现象（比如 `president` 找回同类职位），轮数太少往往只会返回高频噪词。